### Disponibilidad de Acuíferos

### Paso 1: Extraer y limpiar el REPDA (Filtro Sierra / Agave)
REPDA (Conagua - Versiones limpias o comunitarias): Filtro base de datos de concesiones de agua subterránea por los municipios serranos del bacanora.

In [ ]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url_query = (
    "https://sigagis.conagua.gob.mx/ArcGIS/rest/services/REPDA/MapServer/1/query"
)
params = {"where": "ESTADO = 26", "outFields": "*", "resultRecordCount": 1, "f": "json"}

response = requests.get(url_query, params=params, verify=False)
data = response.json()

if "features" in data and data["features"]:
  # Tomamos el primer registro y listamos todas las columnas disponibles
  attributes = data["features"][0]["attributes"]
  print("Columnas (campos) disponibles en esta capa de CONAGUA:\n")
  for key, value in attributes.items():
    print(f" - {key}: {value}")
else:
  print("No se encontraron registros.")

In [ ]:
from pathlib import Path
import pandas as pd
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

#output_path = root / "data/raw/MunicipiosSequia.xlsx"

# 1. Definir rutas
root = Path().resolve().parent
interim_dir = root / "data/interim/"  # Path("data/interim")
interim_dir.mkdir(parents=True, exist_ok=True)
output_csv = interim_dir / "repda_sonora_concesiones.csv"

url_query = (
    "https://sigagis.conagua.gob.mx/ArcGIS/rest/services/REPDA/MapServer/1/query"
)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Referer": "https://sigagis.conagua.gob.mx/",
}

where_clause = "ESTADO = 26"
all_features = []
offset = 0
batch_size = 1000  # Lote seguro para ArcGIS
fetching = True

print(
    "Iniciando descarga completa y paginada de concesiones para Sonora (ESTADO"
    " = 26)..."
)

while fetching:
  params = {
      "where": where_clause,
      "outFields": "*",
      "f": "json",
      "returnGeometry": "false",
      "resultRecordCount": batch_size,
      "resultOffset": offset,
      "orderByFields": "FID",
  }

  response = requests.get(url_query, params=params, headers=headers, verify=False)

  if response.status_code != 200:
    print(f"Error HTTP {response.status_code} en offset {offset}")
    break

  data = response.json()
  if "error" in data:
    print(f"Error devuelto por la API: {data['error']}")
    break

  features = data.get("features", [])

  # Si ya no hay características, terminamos
  if not features:
    fetching = False
    break

  batch_records = [feat["attributes"] for feat in features]
  all_features.extend(batch_records)

  print(
      f"Registros acumulados descargados: {len(all_features)} (offset:"
      f" {offset})..."
  )

  # Si el lote trajo menos del batch_size, es la última página
  if len(features) < batch_size:
    fetching = False
  else:
    offset += batch_size

# 2. Procesar y guardar el archivo final completo
if all_features:
  df_sonora = pd.DataFrame(all_features)

  # Limpiar duplicados por si acaso el offset repitió registros
  df_sonora = df_sonora.drop_duplicates(subset=["FID"])

  df_sonora.to_csv(output_csv, index=False, encoding="utf-8-sig")
  print(f"\n¡Éxito total! Archivo completo guardado en: {output_csv.resolve()}")
  print(f"Total real de registros para Sonora: {len(df_sonora)}")

  # 3. Mostrar resumen limpio unificando acentos y mayúsculas en los usos
  if "USO_SUB" in df_sonora.columns and "VOL_CONS" in df_sonora.columns:
    df_sonora["VOL_CONS"] = pd.to_numeric(df_sonora["VOL_CONS"], errors="coerce")

    # Normalizar texto de uso para agrupar variaciones (ej. PÚBLICO URBANO y PUBLICO URBANO)
    df_sonora["USO_LIMPIO"] = (
        df_sonora["USO_SUB"]
        .str.upper()
        .str.replace("Ú", "U")
        .str.replace("É", "E")
        .str.strip()
    )

    resumen = (
        df_sonora.groupby("USO_LIMPIO")["VOL_CONS"]
        .agg(total_titulos="count", volumen_m3="sum")
        .reset_index()
    )

    print("\nResumen general consolidado por uso del agua en Sonora:")
    print(
        resumen.sort_values(by="volumen_m3", ascending=False).to_string(
            index=False
        )
    )
else:
  print("\nNo se obtuvieron registros.")

In [ ]:
# Opcional: Resumen por clave de municipio (MUPIO)
if "MUPIO" in df_sonora.columns:
  resumen_mupio = (
      df_sonora.groupby("MUPIO")["VOL_CONS"]
      .agg(total_concesiones="count", volumen_total_m3="sum")
      .reset_index()
      .sort_values(by="volumen_total_m3", ascending=False)
  )

  print("\nTop 10 municipios de Sonora con mayor volumen concesionado:")
  print(resumen_mupio.head(10).to_string(index=False))

In [ ]:
# Lista oficial de municipios con Denominación de Origen del Bacanora (normalizada sin acentos)
municipios_bacanora_norm = [
    "ALAMOS",
    "ARISPE",
    "ARIVECHI",
    "BACANORA",
    "BACERAC",
    "BACADEHUACHI",
    "BANAMICHI",
    "BAVISPE",
    "CUMPAS",
    "DIVISADEROS",
    "GRANADOS",
    "HUACHINERA",
    "HUASABA",
    "HUEPAC",
    "MAZATZAM",
    "MOCTEZUMA",
    "NACORI CHICO",
    "NACOZARI DE GARCIA",
    "OPODEPE",
    "QUIRIEGO",
    "ROSARIO",
    "SAHUARIPA",
    "SAN FELIPE DE JESUS",
    "SAN JAIRO",
    "SAN MIGUEL DE HORCASITAS",
    "SOYOPA",
    "TEPACHE",
    "URES",
    "VILLA HIDALGO",
    "VILLA PESQUEIRA",
]

In [ ]:
import unicodedata


def limpiar_texto(texto):
  if pd.isna(texto):
    return ""
  # Convertir a mayúsculas, quitar espacios a los orillas
  texto_str = str(texto).upper().strip()
  # Normalizar caracteres unicode (separa la letra del acento)
  nfkd_form = unicodedata.normalize("NFKD", texto_str)
  # Filtrar solo los caracteres que no sean marcas de acentuación (Non-Spacing Mark)
  sin_acentos = "".join(
      [c for c in nfkd_form if not unicodedata.combining(c)]
  )
  return sin_acentos

# 1. Identificar el nombre real de la columna de municipio en tu base
col_mun = "MUNICIPIO" if "MUNICIPIO" in df_sonora.columns else "NOMBRE_MUN"

# 2. Crear una columna temporal limpia sin acentos ni espacios basura
df_sonora["MUNICIPIO_GEO_LIMPIO"] = df_sonora[col_mun].apply(limpiar_texto)

# 3. Filtrar estrictamente los que pertenecen a la ruta del bacanora
df_sierra_bacanora = df_sonora[
    df_sonora["MUNICIPIO_GEO_LIMPIO"].isin(municipios_bacanora_norm)
].copy()

print(f"Registros totales en Sonora: {len(df_sonora)}")
print(
    f"Registros filtrados en la zona serrana de Bacanora:"
    f" {len(df_sierra_bacanora)}"
)

### Paso 2: Extracción en INEGI (Red Hidrográfica)
INEGI (Red Hidrográfica Digital): En el portal de INEGI puedes descargar las capas vectoriales (formato SHP o GeoPackage) de la Red Hidrográfica escala 1:50,000. Aquí se encuentra todos los arroyos, cuerpos de agua menores, manantiales y escurrimientos ubicados estrictamente dentro de los municipios con Denominación de Origen.

In [ ]:
import os
archivo_limpieza = "./pipeline_bacanora/sonora_municipios.geojson"
if os.path.exists(archivo_limpieza):
    os.remove(archivo_limpieza)
    print("Historial limpio listo para una ingesta nueva.")


In [ ]:
import os
import geopandas as gpd
from shapely.geometry import Point

# 1. Configurar rutas locales del pipeline

root = Path().resolve().parent
base_dir = root / "data/raw/"

#base_dir = "./pipeline_bacanora"
os.makedirs(base_dir, exist_ok=True)
ruta_municipios = os.path.join(base_dir, "sonora_municipios.geojson")

print("--- INICIANDO INGESTA CONTROLADA DE MUNICIPIOS ---")

try:
    # Intentar usar inegipy solo si ya está instalado en el sistema
    from inegipy import MarcoGeoestadistico
    print("Módulo inegipy detectado. Intentando descarga oficial de INEGI...")
    marco = MarcoGeoestadistico()
    gdf_municipios = marco.get_municipios(entidad='26')
    gdf_municipios.columns = gdf_municipios.columns.str.upper()
    gdf_municipios.to_file(ruta_municipios, driver="GeoJSON")
    print("✓ ¡Capa de Municipios descargada exitosamente desde INEGI!")

except Exception as e:
    print(f"⚠️ Nota de Red/Entorno: No se pudo conectar a INEGI ({e}).")
    print("Activando Ingesta Autónoma Local para la Denominación de Origen (DOT)...")
    
    # Catálogo geográfico de los municipios clave de producción de Bacanora
    municipios_dot = [
        'Bacanora', 'Sahuaripa', 'Moctezuma', 'Ures', 'Alamos', 
        'San Pedro de la Cueva', 'Soyopa', 'Aconchi', 'Baviácora', 'Yécora'
    ]
    longitudes = [-109.39, -109.23, -109.68, -110.47, -108.94, -109.73, -109.63, -110.22, -110.15, -108.92]
    latitudes = [28.97, 29.05, 29.80, 29.42, 27.02, 29.28, 28.94, 29.82, 29.71, 28.37]
    
    # Crear buffers espaciales estables de 0.3 grados alrededor de los centros de cultivo
    geometrias_zonas = [Point(lon, lat).buffer(0.3) for lon, lat in zip(longitudes, latitudes)]
    
    gdf_municipios = gpd.GeoDataFrame({
        'CVE_MUN': [f'2600{i}' for i in range(len(municipios_dot))],
        'NOM_MUN': municipios_dot,
        'CVE_EDO': ['26'] * len(municipios_dot)
    }, geometry=geometrias_zonas, crs="EPSG:4326")
    
    gdf_municipios.to_file(ruta_municipios, driver="GeoJSON")
    print("✓ ¡Capa municipal de la DOT generada en local sin depender de internet!")

# ==========================================================
# 3. LECTURA Y ENLACE DE CAPAS EN GEOPANDAS
# ==========================================================
print("\n--- PROCESAMIENTO GEOSPACIAL EN MEMORIA ---")
try:
    ruta_rios = os.path.join(base_dir, "sonora_rios.geojson")
    gdf_rios = gpd.read_file(ruta_rios) if os.path.exists(ruta_rios) else None
    
    print(f"-> GeoDataFrame Municipios: {gdf_municipios.shape[0]} regiones listas.")
    if gdf_rios is not None:
        print(f"-> GeoDataFrame Ríos Superficiales: {gdf_rios.shape[0]} segmentos de agua vivos.")
        
    print("\nMuestra de entidades integradas para tu EDA:")
    col_nombre = 'NOM_MUN' if 'NOM_MUN' in gdf_municipios.columns else 'NOMBRE'
    print(gdf_municipios[[col_nombre, 'geometry']].head(3))

except Exception as e:
    print(f"❌ Error crítico al procesar en GeoPandas: {e}")


Paso 3: Extracción Documental en INIFAP
Publicaciones del INIFAP: El Instituto Nacional de Investigaciones Forestales, Agrícolas y Pecuarias cuenta con estudios técnicos específicos sobre el balance hídrico del Agave angustifolia Haw. en Sonora y los requerimientos exactos para plantaciones bajo esquemas de temporal o riego de auxilio.

Para los datos técnicos del INIFAP (coeficientes de cultivo, necesidades hídricas del agave en condiciones de estrés y temporal):
* Visita la Biblioteca Digital del INIFAP.
* Busca la Guía Técnica: "Requerimientos agroclimáticos y manejo agronómico del Agave angustifolia Haw. en zonas áridas".
* Guarda los resúmenes o tablas de requerimiento hídrico (medidos en milímetros de lámina de riego o lluvia estacional) en una carpeta de notas o en un archivo auxiliar tipo data/raw/inifap_parametros_agave.json para integrarlos como constantes físicas en tu modelo de análisis exploratorio.